# Notebook 07 — Random Numbers and Reproducibility

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 07 of 20  
**Prerequisites:** Notebook 06  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

A result that cannot be reproduced is not a result — it's an observation.
Random number generation is the most common source of irreproducibility in
computational biology: different seeds, different splits, different permutations,
different conclusions.

This notebook establishes the reproducibility contract used everywhere in this programme.

---
## Step 2 — Intuition

A pseudo-random number generator (PRNG) is a deterministic function: given the same
seed, it produces the same sequence of numbers every time. 'Random' means 'hard to
predict without knowing the seed', not 'physically random'.

The new NumPy API (`np.random.default_rng(seed)`) creates an isolated generator.
The old API (`np.random.seed(42)`) sets a global state — dangerous in multi-threaded
or modular code because any import can reset it.

---
## Step 3 — Biological Background

**Where randomness appears in computational biology:**

| Context | Random element | Reproducibility requirement |
|---------|---------------|-----------------------------|
| Train/test split | Which samples go to test set | Fixed seed per experiment |
| k-means clustering | Initial centroid positions | Fixed seed; report seed in paper |
| Bootstrap confidence intervals | Resample indices | Fixed seed; report n_bootstrap |
| Stochastic simulation | Gillespie algorithm events | Fixed seed; verify with multiple seeds |
| Data augmentation | Noise added to training data | Fixed seed per epoch |

In a paper: always report the seed. A result that only holds for one seed is not robust.

---
## Step 4 — Mathematical Explanation

**Bootstrap confidence interval for the mean:**

Given data $x_1, \ldots, x_n$, draw $B$ bootstrap samples $x^{*(b)}$ of size $n$
with replacement. Compute the statistic $\hat{\theta}^{*(b)}$ for each.
The 95% CI is the $2.5^{\text{th}}$ and $97.5^{\text{th}}$ percentiles of
$\{\hat{\theta}^{*(1)}, \ldots, \hat{\theta}^{*(B)}\}$.

This is non-parametric — no distributional assumption is made.

---
## Step 5 — Computational Explanation

**NumPy Generator API (use this, not `np.random.seed`):**

```python
rng = np.random.default_rng(seed=42)  # creates an isolated generator
rng.standard_normal(size=(100, 5))    # normal variates
rng.integers(0, 10, size=20)          # uniform integers
rng.choice(arr, size=50, replace=True) # bootstrap
rng.permutation(n)                    # random permutation
```

Pass `rng` as a parameter to any function that needs randomness — never create
a global generator inside a function.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np

# Cell 6.1 — Old vs new API
# Old (global state — avoid in library code):
np.random.seed(42)
old_sample = np.random.standard_normal(5)

# New (isolated generator — use this):
rng = np.random.default_rng(seed=42)
new_sample = rng.standard_normal(5)

print("Old API:", old_sample.round(4))
print("New API:", new_sample.round(4))
print("(Different values from same seed — different algorithm)'")

In [ ]:
# Cell 6.2 — Reproducibility: same seed → same output
def make_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)

run1 = make_rng(42).standard_normal(10)
run2 = make_rng(42).standard_normal(10)
run3 = make_rng(99).standard_normal(10)

print(f"Same seed, run 1 vs 2: {np.allclose(run1, run2)}")
print(f"Different seed (42 vs 99): {np.allclose(run1, run3)}")

In [ ]:
# Cell 6.3 — Bootstrap confidence interval for GC content
def bootstrap_ci(
    data: np.ndarray,
    statistic,
    n_bootstrap: int = 2000,
    confidence: float = 0.95,
    rng: np.random.Generator | None = None,
) -> tuple[float, float, float]:
    """
    Percentile bootstrap confidence interval.

    Returns
    -------
    (point_estimate, lower, upper)
    """
    if rng is None:
        rng = np.random.default_rng()
    point = statistic(data)
    boots = [
        statistic(rng.choice(data, size=len(data), replace=True))
        for _ in range(n_bootstrap)
    ]
    alpha = (1 - confidence) / 2
    lower, upper = np.quantile(boots, [alpha, 1-alpha])
    return point, lower, upper

# Synthetic GC content measurements from 50 windows
rng_main = np.random.default_rng(42)
gc_values = rng_main.beta(a=6, b=4, size=50)  # Beta(6,4) → mean ~0.6

est, lo, hi = bootstrap_ci(gc_values, np.mean, n_bootstrap=5000, rng=rng_main)
print(f"GC content estimate: {est:.4f}")
print(f"95% Bootstrap CI:    [{lo:.4f}, {hi:.4f}]")

In [ ]:
# Cell 6.4 — Reproducible train/test split for expression data
def stratified_split(
    n: int,
    labels: np.ndarray,
    test_frac: float = 0.2,
    rng: np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Stratified train/test split preserving class proportions."""
    if rng is None:
        rng = np.random.default_rng()
    train_idx, test_idx = [], []
    for cls in np.unique(labels):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)
        n_test = max(1, int(len(cls_idx) * test_frac))
        test_idx.extend(cls_idx[:n_test])
        train_idx.extend(cls_idx[n_test:])
    return np.array(train_idx), np.array(test_idx)

labels = np.array([0]*40 + [1]*40 + [2]*20)
rng2 = np.random.default_rng(seed=42)
train, test = stratified_split(100, labels, rng=rng2)
print(f"Train: {len(train)} samples, Test: {len(test)} samples")
print(f"Test class distribution: {np.bincount(labels[test])}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Bootstrap distribution visualization
import matplotlib.pyplot as plt

rng_plot = np.random.default_rng(42)
gc_data = rng_plot.beta(6, 4, size=50)
boot_means = [
    rng_plot.choice(gc_data, size=50, replace=True).mean()
    for _ in range(5000)
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(boot_means, bins=50, color="steelblue", alpha=0.7, edgecolor="white")
lo, hi = np.quantile(boot_means, [0.025, 0.975])
ax.axvline(np.mean(gc_data), color="black", lw=2, label=f"Estimate = {np.mean(gc_data):.3f}")
ax.axvline(lo, color="tomato", lw=1.5, linestyle="--", label=f"95% CI [{lo:.3f}, {hi:.3f}]")
ax.axvline(hi, color="tomato", lw=1.5, linestyle="--")
ax.set_xlabel("Bootstrap mean GC content")
ax.set_ylabel("Count")
ax.set_title("Bootstrap distribution of mean GC content (B=5000)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Add `bootstrap_ci` to `compbio_utils/stats.py`. Make `rng` a parameter with a
   sensible default.
2. Run `bootstrap_ci` with seeds 1, 2, 3, 42, 99 and report the five CI estimates.
   Are they all similar? What does that tell you about B=5000?
3. Why does `np.random.seed(42)` inside a library function violate encapsulation?
   Write a test that demonstrates the problem.
4. Add the `seed` used in each analysis notebook to a `REPRODUCIBILITY.md` stub in
   the module folder.

---
## Quiz — Active Recall

1. What is the difference between `np.random.seed(42)` and `np.random.default_rng(42)`?
2. A colleague says their k-means analysis is reproducible because they "always use seed 42."
   Name one way their result could still be non-reproducible.
3. What distribution does `rng.beta(a=6, b=4)` have? What is its expected value?
4. In `bootstrap_ci`, why is `rng` passed as a parameter rather than created inside the function?
5. What is the percentile bootstrap CI? What assumption does it NOT make?

---
## Reflection

**Date completed:** ____________________

> *[Can you write bootstrap_ci from scratch? Have you verified it gives the same output for the same seed? Is it in compbio_utils?]*

---
*Next: `08_pandas_fundamentals.ipynb`*